# <font color='green'><b><u>Single-cell downstream report<u></b></font>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc

In [ ]:
path_adata = FILE
selected_clustering = CLUSTERING_NAME

In [ ]:
adata = sc.read(path_adata)

In [ ]:
from IPython.display import display, HTML

display(HTML("<p>After filtering, the data set contains " + str(adata.n_obs) + " cells and " + str(adata.n_vars) 
             + " genes or transcripts. The number of cells per sample are listed below:</p>"))

In [ ]:
adata.obs.value_counts('sample').rename_axis('samples').reset_index(name='number of cells')

---
## <font color='green'>Predicted labels</font>

Celltypist uses pre-trained logistic regression models for predicting labels such as cell types for each cell. Predicted labels of the selected model(s) and the confidence of the predictions are displayed in the UMAPs below. Counts for the predicted labels are summarised in the table(s).

In [ ]:
predicted_labels = [ colname for colname in adata.obs.columns if colname.startswith('celltypist:') ]
for label in predicted_labels:
    sc.pl.umap(adata, color=label)

In [ ]:
from IPython.display import display, HTML
import base64

for label in predicted_labels:
    if not label.endswith(':conf'):
        html = f'<font color="black"><details><summary>Click to expand {label} table</summary>'
        df = adata.obs.value_counts(label).rename_axis(label).reset_index(name='number of cells')
        html += '<div style="overflow-y: scroll; overflow-x: scroll; max-height: 400px">' + df.to_html() + '</div></details>'
        display(HTML(html))

---
## <font color='green'>Clusters</font>

In [ ]:
import numpy as np
from IPython.display import display, HTML

display(HTML(f"Cells are grouped by {selected_clustering}. The number of cells per cluster are listed below:"))

In [ ]:
adata.obs.value_counts(selected_clustering).rename_axis(selected_clustering).reset_index(name='number of cells')

In [ ]:
sc.pl.umap(adata, color=[selected_clustering])

In [ ]:
if 'X_umap_scvi' in adata.obsm:
    sc.pl.embedding(adata, basis='X_umap_scvi', color=[selected_clustering], title='scvi UMAP')

In [ ]:
# define order of clusters numerically
import numpy as np

list_clusters = np.unique(adata.obs[selected_clustering])
list_clusters = sorted([int(x) for x in list_clusters])
list_clusters = [str(x) for x in list_clusters]

In [ ]:
# overlap with annotation
import pandas as pd
import anndata as ann

def create_overlap_pct_label(adata: ann.AnnData, list_clusters: list, clustering: str, annotation: str, 
                             ntop: int=5, min_overlap_ratio: float=0.05) -> pd.DataFrame:
    """
    Computes the overlap with a given annotation for each cluster

    Parameters
    ----------
    adata: ann.AnnData
        AnnData object containing the clusters and annotation labels
    list_clusters: list 
        A list of cluster IDs defining the order of rows in the output table
    clustering: str
        Name of clustering entry
    annotation: str
        Name of the annotation, e.g. celltypist annotation
    ntop: int
        Number of top annotations to be displayed
    min_overlap_ratio: float
        Minimum ration of cells to consider annotation label for output

    Returns
    -------
    pd.DataFrame
        Table listing the top N annotations per cluster including the percentage of cells
    """
    dict_clus_overlap = {}

    for clus in list_clusters:
        size = adata[ adata.obs[clustering] == clus ].shape[0]
        list_topn = adata[ adata.obs[clustering] == clus ].obs.value_counts(annotation)[0:ntop]
        
        ratios = list_topn / size
        ratios = ratios[ ratios >= min_overlap_ratio]
        list_str = [ label + ' (' + str(round(ratio* 100, 2)) + '%)' for label, ratio in zip(ratios.index, ratios.values) ]
        
        if len(list_str) < ntop:
            nmissing = ntop - len(list_str)
            list_str = list_str + nmissing * ['-']
        dict_clus_overlap[clus] = list_str

    df_overlap = pd.DataFrame(dict_clus_overlap).T
    df_overlap.index.name = 'Cluster'
    df_overlap.columns = ['top ' +  str(i+1) for i in list(range(ntop)) ]

    return df_overlap

In [ ]:
from IPython.display import display, HTML
import base64

for label in predicted_labels:
    if not label.endswith(':conf'):
        
        df_overlap = create_overlap_pct_label(adata, list_clusters, selected_clustering, label)
        
        html = "<h3><font color='green'>Cluster annotation - " + label
        html += '</font></h3><font color="black"><details><summary>Click to expand table</summary>'
        html += '<div style="overflow-y: scroll; overflow-x: scroll; max-height: 400px">' + df_overlap.to_html() + '</div></details>'
        display(HTML(html))

---
## <font color='green'><b>Marker genes</b></font>

Marker genes per clusters are identified based on a Wilcoxon rank-sum test by comparing the gene expression between cells of a cluster against all other cells.

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby=selected_clustering, standard_scale="var", swap_axes=True, n_genes=3)

In [ ]:
import numpy as np
from IPython.display import display, HTML


for clus in list_clusters:
    display(HTML("<h3><font color='green'>Markers cluster " + clus + 
                 "</font></h3><details><div style='overflow-y: scroll; max-height: 400px'>" + 
             sc.get.rank_genes_groups_df(adata, group=clus).head(100).to_html() + "</div></details>"))

---
## <font color='green'><b>Gene Set Enrichment</b></font>

g:Profiler gene set enrichment per cluster. Cluster marker genes are further filtered by their expression inside and outside of the cluster before computing enrichment.

In [ ]:
import numpy as np
from IPython.display import display, HTML


for clus in list_clusters:
    display(HTML("<h3><font color='green'>Enrichment cluster " + clus + 
                 "</font></h3><details><div style='overflow-y: scroll; overflow-x: scroll; max-height: 400px'>" + 
             adata.uns['rank_genes_groups']['enrich'][clus].head(100).to_html() + "</div></details>"))

---

## <font color='green'><b>Cell-cell interaction</b></font>

LIANA+ consensus ligand-receptor interactions. LIANA+ uses ranking and aggregating to compute a `magnitude rank`. A lower `magnitude rank` indicates that an interaction is more likely to occur.

Following methods are currently implemented in the LIANA+ framework:

In [ ]:
import liana as li

li.mt.show_methods()

In [ ]:
# create images
import numpy as np
import matplotlib.pyplot as plt
import math

ntop = 20

paths_images = {}

if 'liana_res' in adata.uns:
    for clus in list_clusters:
        
        path_img = f"./dotplot_test_clus{clus}.png"
        paths_images[clus] = path_img
                        
        fig = li.pl.dotplot(adata = adata, 
              colour='magnitude_rank',
              size='specificity_rank',
              inverse_size=True,
              inverse_colour=True,
              source_labels=[clus],
              target_labels=list_clusters,
              top_n=ntop, 
              orderby='magnitude_rank',
              orderby_ascending=True,
              return_fig=True)

        fig.save(path_img, bbox_inches='tight')

In [ ]:
# HTML output
from IPython.display import display, HTML
import base64

if 'liana_res' in adata.uns:
    df_liana = adata.uns['liana_res'].copy()
    
    for clus, path in paths_images.items():
        with open(path, "rb") as f_plot:
            # dotplot
            image_string = base64.b64encode(f_plot.read()).decode("utf-8")
            image_html = f'<h3><font color="green">Interactions cluster {clus}</h3></font>'
            image_html += '<details><summary>Click to expand dotplot</summary>'
            image_html += f'<figure><img alt="liana image" src="data:image/png;base64,{image_string}" />'
            image_html += f'<figcaption>Shown are the top {ntop} interactions between cluster {clus} and the other clusters. '
            image_html += 'The color gradient and dot size indicate the -log10 <i>magnitude rank</i> and <i>specificity rank</i>, respectively. Higher values are better.</figcaption></figure></details>'
            
            # table
            df_liana_clus = df_liana[ df_liana['source'] == clus]
            image_html += '<font color="black"><details><summary>Click to expand table</summary>'
            image_html += "<div style='overflow-y: scroll; overflow-x: scroll; max-height: 400px'>" + df_liana_clus.head(100).to_html() + "</div></details>"
            
            display(HTML(image_html))

---

## <font color='green'><b>Citations</b></font>

### <font color='green'>[Nextflow](https://www.nextflow.io/)</font>

_Di Tommaso P, Chatzou M, Floden EW, Barja PP, Palumbo E, Notredame C. (2017). Nextflow enables reproducible computational workflows. Nat Biotechnol., 35(4):316-319._

### <font color='green'>[Nfcore](https://nf-co.re/)</font>
This pipelines is a substantially modified version of [nfcore:scdowntream](https://github.com/nf-core/scdownstream), which was originally written by Nico Trummer, and uses [nfcore modules](https://github.com/nf-core/modules): 

_Ewels PA, Peltzer A, Fillinger S, Patel H, Alneberg J, Wilm A, Garcia MU, Di Tommaso P, Nahnsen S. (2020). The nf-core framework for community-curated bioinformatics pipelines. Nat Biotechnol., 38(3):276-278_

### <font color='green'>[Scanpy](https://scanpy.readthedocs.io/en/stable/)</font>

_Wolf FA, Angerer P, & Theis, FJ. (2018). SCANPY: large-scale single-cell gene expression data analysis. Genome Biol., 19(1):15._

### <font color='green'>[Leiden clustering](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.leiden.html)</font>
_Traag VA, Waltman L, & Van Eck NJ. (2019). From Louvain to Leiden: guaranteeing well-connected communities., Sci. Rep., 9(1):5233._

### <font color='green'>[scVI](https://scvi-tools.org/)</font>
_Lopez R, Regier J, Cole MB, Jordan MI, & Yosef N. (2018). Deep generative modeling for single-cell transcriptomics. Nature methods, 15(12):1053-1058._

### <font color='green'>[Celltypist](https://www.celltypist.org/)</font>
_Domínguez Conde C, Xu C, Jarvis LB, Rainbow DB, Wells SB, Gomes T, ... & Teichmann, SA (2022). Cross-tissue immune cell analysis reveals tissue-specific features in humans. Science, 376(6594):eabl5197._

### <font color='green'>[gProfiler](https://biit.cs.ut.ee/gprofiler/gost)</font>
_Kolberg L, Raudvere U, Kuzmin I, Adler P, Vilo J, & Peterson H. (2023). g: Profiler—interoperable web service for functional enrichment analysis and gene identifier mapping (2023 update). Nucleic Acids Res., 51(W1):W207-W212._

### <font color='green'>[LIANA+](https://github.com/saezlab/liana-py)</font>

_Dimitrov D, Schäfer PSL, Farr E, Rodriguez-Mier P, Lobentanzer S, Badia-i-Mompel P, ... & Saez-Rodriguez J. (2024). LIANA+ provides an all-in-one framework for cell–cell communication inference. Nat. Cell Biol., 26(9):1613-1622._
